# Shadow Agent Pro — Merge URLhaus + Tranco (Colab)

Merges your downloaded `csv.txt` (URLhaus malicious URLs) and `top-1m.csv`
(Tranco top domains) into the `url,label` format the project's `train.py`
expects. Runs entirely standalone — no need to upload the rest of the
project for this step.

## 1. Upload your two files
Click the cell below, then **Choose Files** and select both `csv.txt` and `top-1m.csv`.

In [ ]:
from google.colab import files
uploaded = files.upload()

## 2. Merge them

In [ ]:
import csv
import random

random.seed(42)
MAX_PER_CLASS = 10000  # change this if you want a bigger/smaller dataset

# --- parse URLhaus (malicious) ---
with open("csv.txt", encoding="utf-8", errors="replace") as f:
    text = f.read()

lines = [line for line in text.splitlines() if not line.startswith("#")]
reader = csv.reader(lines)

malicious_urls = []
for row in reader:
    if len(row) < 3:
        continue
    url = row[2].strip().strip('"')
    if url and url.startswith(("http://", "https://")):
        malicious_urls.append(url)

print(f"Parsed {len(malicious_urls):,} malicious URLs from URLhaus")

# --- parse Tranco (benign) ---
with open("top-1m.csv", encoding="utf-8", errors="replace") as f:
    text = f.read()

benign_domains = []
for line in text.splitlines():
    parts = line.strip().split(",")
    if len(parts) == 2 and parts[1]:
        benign_domains.append(parts[1].strip())

benign_urls = [f"http://{d}" for d in benign_domains]
print(f"Parsed {len(benign_urls):,} domains from Tranco")

# --- merge, shuffle, cap, write ---
random.shuffle(malicious_urls)
random.shuffle(benign_urls)

if MAX_PER_CLASS > 0:
    malicious_urls = malicious_urls[:MAX_PER_CLASS]
    benign_urls = benign_urls[:MAX_PER_CLASS]

rows = [(u, 1) for u in malicious_urls] + [(u, 0) for u in benign_urls]
random.shuffle(rows)

with open("urls_labeled_live.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["url", "label"])
    writer.writerows(rows)

print(f"\nWrote {len(rows):,} rows ({len(benign_urls):,} benign, {len(malicious_urls):,} malicious) to urls_labeled_live.csv")

## 3. Sanity-check the output before downloading

In [ ]:
import pandas as pd
df = pd.read_csv("urls_labeled_live.csv")
print(f"Total rows: {len(df)}")
print(df["label"].value_counts())
df.head()

## 4. Download the merged file back to your computer

After this, place `urls_labeled_live.csv` at `ml-training/datasets/urls_labeled_live.csv`
in your project, then run (on your own machine, not in Colab):
```
cd ml-training
python train.py --data datasets/urls_labeled_live.csv --out ../backend/ml/artifacts
```

In [ ]:
from google.colab import files
files.download("urls_labeled_live.csv")